# OPAA di-Zn phosphotriesterase — TS pipeline (ordered protocol)

The recommended in-order protocol for the OPAA di-Zn theozyme (SN2 at phosphorus), a
charged metal active site, synthesized from a committee review (3 independent agents +
codex) of the docs + code. A focused subset of the generalized notebook with OPAA inputs
pre-filled — **edit the atom serials and input PDB for your structure** (only example
values are filled; nothing is hardcoded).

**What "path search" vs "refine to a saddle" means** (a question worth nailing): a
**path search** *proposes* a TS GUESS along the reaction coordinate — the **1-D relaxed
scan** here IS a (single-ended) path search, and it hands you a TS guess **plus**
approximate reactant/product endpoints. **`refine-ts` is a SEPARATE step**: it optimizes
that guess to an *exact* first-order saddle and runs the Hessian gate — it does NOT search
a path. So you need a guess (scan or NEB) *before* refine-ts. And yes — you then
**energy-minimize the endpoints** to true basins, because the barrier is
`E(TS) − E(reactant_min)`.

**Protocol:** `0` protonate (no PTM) → `1` monitor (--metals) → `2` reaction-spec
(O_nuc→P forming, P→O_lg breaking) → `3` CA-frozen relax of the cluster (reactant basin) →
`4` 1-D relaxed scan → TS guess **+ R/P endpoint frames** → `5` **minimize the R & P
endpoints** (true basins) → `6` *(optional)* CI-NEB between the minima (more rigorous guess
for an asynchronous step) → `7` refine-ts (`--backend auto`, 1 imaginary mode) →
`8` validate-ts (tier b, Zn-shell active region) → `9` verify-irc-like → `10` optional ORCA DFT.

**Model:** default **`mace-polar-m`** (MACE-POLAR-1-M) — polarizable + long-range
electrostatics, ideal for the charged di-Zn pocket, and **baked into MAIN_SIF so it loads
in-process**. `mace-omol` is the higher-accuracy second pass (large GPU); `mace-mh-1 --head
omol` and `orb-mol-conservative` are charge-aware alternatives; UMA/eSEN route to UMA_SIF.
**Never** GFN2-xTB on the metals. **Don't** use React-OT/AEFM here — they are CHNO/gas-phase
only and HARD-FAIL on Zn/P.

**Charge & multiplicity (pre-filled):** net charge of the protonated system = **0** (no PTM);
no radicals → spin quantum number **S = 0** → spin **multiplicity M = 2S+1 = 1** (singlet), so
pass **`--multiplicity 1`**. (The codebase uses *multiplicity* (2S+1) everywhere; the optional
`--spin` flag instead takes **S** and converts it to 2S+1, so `--spin 0` ⇒ `--multiplicity 1`.)
Charge & multiplicity are **CLI-only** — the reaction-spec YAML ignores them.

**Watch for a pentacoordinate intermediate.** Organophosphate hydrolysis at P is often
*stepwise* through a trigonal-bipyramidal phosphorane (both P–O bonds ~1.7 Å, CV s≈0). If
`verify-irc-like` lands in such a basin and an all-real Hessian confirms it's a minimum, it's
a real intermediate — split into two TS searches (R→intermediate, intermediate→P).

# **NOTEBOOK INITIALIZATION**

In [36]:
# ═══════════════════════════════════════════════════════════════════════════════
#  NOTEBOOK INITIALIZATION — run this cell at the start of every session
# ═══════════════════════════════════════════════════════════════════════════════

PROJECT_NAME = 'opaa_theozyme'

# > USER CONFIGURATION
# >> project paths
HOME_DIR      = '/home/woodbuse/'
THEOZYME_DIR  = f'{HOME_DIR}codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/theozyme/'
SYSTEM_DIR    = THEOZYME_DIR        # the active-site / structure working dir

# >> manual overrides (set to None to use defaults)
_WORKING_DIR_OVERRIDE = f'{HOME_DIR}codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme'   # default: notebook directory
_OUTPUT_DIR_OVERRIDE  = f'{HOME_DIR}codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/'   # default: WORKING_DIR/output/

# >> subdirectories to create (each becomes an UPPERCASE <NAME>_DIR global)
WORKING_SUBDIRS = ['cmds', 'submit', 'logs', 'FINAL']
OUTPUT_SUBDIRS  = ['protomers', 'monitor', 'reaction_spec', 'relax_minimize', 'scan', 'path_search', 'ts_search', 'generative',
                   'refine_ts', 'ts_validation', 'dft']

# > IMPORTS
# >> standard library
import concurrent.futures, copy, glob, itertools, json, math, multiprocessing, os, operator
import random, re, shlex, shutil, statistics, string, subprocess, sys, textwrap, time, warnings
from collections import Counter, defaultdict, OrderedDict, deque
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed, FIRST_COMPLETED, wait
from datetime import datetime
from itertools import permutations, product, islice
from math import log10, floor
from pathlib import Path
from pprint import pprint
# >> third-party: data & math
import numpy as np, pandas as pd
from scipy.stats import gaussian_kde
# >> third-party: plotting
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import to_rgba, ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
# >> third-party: structural biology
import pyrosetta, pyrosetta.distributed.tasks.rosetta_scripts as rosetta_scripts
from Bio import PDB
from Bio.PDB import PDBParser, PPBuilder
from Bio.SeqUtils import seq1
# >> third-party: notebook & misc
from difflib import SequenceMatcher
from IPython.display import display, HTML
# >> custom modules  (notebook_core re-exports the SLURM helpers from slurm_submission)
_NB_FUNCS_PATH = f'{HOME_DIR}special_scripts/notebook_functions'
if _NB_FUNCS_PATH not in sys.path:
    sys.path.insert(0, _NB_FUNCS_PATH)
import notebook_core as nb  # colors, setup_directories, print_initialization, submit_array_job, submit_cpu

# > PATHS
# >> working & output
WORKING_DIR = nb.resolve_working_dir(override=_WORKING_DIR_OVERRIDE, strip_mnt=True)
OUTPUT_DIR  = nb.resolve_output_dir(WORKING_DIR, override=_OUTPUT_DIR_OVERRIDE, strip_mnt=True)

# >> params / ligand files (OK if these don't exist yet)
PARAMS_DIR = f'{SYSTEM_DIR}params/'
CST_DIR    = f'{SYSTEM_DIR}cst_files/'

# >> tools & software
QUANTUM_COWBOY_DIR  = '/home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/'
SPECIAL_SCRIPTS_DIR = f'{HOME_DIR}special_scripts/'
GIT_DIR             = f'{HOME_DIR}git/'
OBABEL_PATH         = f'{HOME_DIR}conda_envs/openbabel_env/bin/obabel'

# >> CONTAINERS (Cowboy Quantum Chemistry) — EDIT to your deployed sif paths.
MAIN_SIF    = '/net/software/containers/users/woodbuse/quantum_chem/quantum_chem-20260604.sif' # MAIN_SIF has the MLFFs (MACE / MACE-OMol / ORB / AIMNet2) + xTB + the `qcb` CLI.
UMA_SIF     = '/net/software/containers/users/woodbuse/quantum_chem/uma-20260527.sif'          # UMA_SIF holds the FairChem models (UMA / eSEN / AllScAIP).
REACTOT_SIF = f'{QUANTUM_COWBOY_DIR}containers/reactot-20260605.sif'                           # The generative MODELS (React-OT proposer, AEFM refiner) each live in their own sidecar.
AEFM_SIF    = f'{QUANTUM_COWBOY_DIR}containers/aefm-20260605.sif'                              # The generative MODELS (React-OT proposer, AEFM refiner) each live in their own sidecar.

def container_for(model):
    """Route an energy-model alias to the sif that can load it (plug-and-play models)."""
    m = (model or '').lower()
    return UMA_SIF if m.startswith(('uma', 'esen', 'allscaip')) else MAIN_SIF

def APPTAINER(sif, gpu=True):
    """apptainer exec prefix (list). --nv for GPU; binds /home + /net."""
    return ['apptainer', 'exec'] + (['--nv'] if gpu else []) + ['--bind', '/home', '--bind', '/net', sif]

# The `cowboy-qc` console-script is NOT baked into the sifs — invoke the CLI as a
# module against the bind-mounted repo (PYTHONPATH) instead. One definition, reused
# everywhere (qcb_cmd / sidecar_cmd / monitor / reaction-spec).
CLI = ['env', f'PYTHONPATH={QUANTUM_COWBOY_DIR}', 'python', '-m', 'quantum_engine.cli']

def qcb_cmd(model, *args, gpu=True):
    """Full cowboy-qc CLI command (list) in the sif that can load `model`."""
    return [*APPTAINER(container_for(model), gpu=gpu), *CLI, *map(str, args)]

def sidecar_cmd(sif, *args, gpu=True):
    """A generative-sidecar command (React-OT / AEFM) in its own sif."""
    return [*APPTAINER(sif, gpu=gpu), *CLI, *map(str, args)]

# > SETUP
for p in [WORKING_DIR, OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)
nb.setup_directories(WORKING_DIR, WORKING_SUBDIRS, export_globals=True, globals_dict=globals())
nb.setup_directories(OUTPUT_DIR,  OUTPUT_SUBDIRS,  export_globals=True, globals_dict=globals())
nb.set_pandas_display(all_on=True)

# > INITIALIZE
os.chdir(WORKING_DIR)
nb.print_initialization(WORKING_DIR, OUTPUT_DIR, project_name=PROJECT_NAME, obabel_path=OBABEL_PATH, globals_dict=globals(), preview=True)
for _n, _s in [('MAIN', MAIN_SIF), ('UMA', UMA_SIF), ('REACTOT', REACTOT_SIF), ('AEFM', AEFM_SIF)]:
    print(f'  CONTAINER {_n:8} {"OK " if Path(_s).exists() else "MISSING"} {_s}')

──────────────────────────────────────────────────────────────────────
  PROJECT: opaa_theozyme
  INITIALIZED: 2026-06-09 at 09:19:34
──────────────────────────────────────────────────────────────────────
  WORKING_DIR = /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/
  OUTPUT_DIR  = /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/
──────────────────────────────────────────────────────────────────────
  EXPORTED SUBDIRS (33 total):
    AF2_OUT_DIR = /net/scratch/woodbuse/organophosphatase/i4_design_260515/af2_out/
    AF3_JSON_DIR = /home/woodbuse/projects/organophosphatase/pxn/design_campaign_i4__pte_hbond_260515/af3_json/
    AF3_OUT_DIR = /net/scratch/woodbuse/organophosphatase/i4_design_260515/af3_out/
    ... and 30 more (set preview=False to show all)
──────────────────────────────────────────────────────────────────────
  CONTAINER MAIN     OK  /net/software/containers/users/woodbuse/quantum_chem/q

# **STEP 0: Protonate / Generate Protomers**

In [30]:
##################################################################
###          PROTONATE STRUCTURE   (cowboy-qc protonator v2)         ###
##################################################################
# Deterministic, staged protonation of the protein in a PDB/CIF.
#   Stage 1  cap open backbone N/C termini (geometry-detected)
#   Stage 2  PTM / covalent safety checks (warnings only)
#   Stage 3  pH-aware canonical protonation
#   Stage 4  histidine tautomer (metal / clash / H-bond geometry)
#   Stage 5  propka refinement of ambiguous states
#   Stage 6  optional multi-protomer output
#   + optional cheap CPU MLFF relax of ONLY the new hydrogens
# HETATM (ligand/metal/water) is assumed already protonated and left alone.
# This cell only PRINTS the command — copy-paste it into your terminal (no sbatch).

print_commands = True

### INPUTS ###
input_pdb = f"{THEOZYME_DIR}input_theozyme/opaa_3l7g_optimal_maximal_theozyme_pxn_unprotonated.pdb"

### OUTPUTS ###
# With protomers > 1 this is the BASE name -> <base>_protomer1..N.pdb
output_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"

### CONSTANTS ###
CONTAINER  = MAIN_SIF       # the cowboy-qc / protonator container (defined in INIT)
PROTONATOR = f"{QUANTUM_COWBOY_DIR}quantum_engine/prep/protonator.py" # (`cowboy-qc protonate <same args>` is equivalent to running this file directly.)

### PROTONATOR PARAMETERS ###
# --- core ----------------------------------------------------------
pH        = 7.5                # target pH for canonical assignment + propka
protomers = 1                  # 1 = single best state; N>1 = N most likely protomers
# --- terminus capping ----------------------------------------------
# N-cap: nh2 (default) | nh3+ | nme | nfo | none     C-cap: cho | coo- | cooh | conh2 | conhme | none
n_cap = "nh2"
c_cap = "cho"
n_cap_overrides = {}            # "CHAIN:RESID": captype
c_cap_overrides = {}
# --- hard protonation overrides (always win over canonical + propka) ---
# "CHAIN:RESID": STATE  (HID HIE HIP ASP ASH GLU GLH LYS LYN CYS CYM CYX TYR TYM ARG)
set_overrides = {}
# --- PTMs (you MUST declare them; residue is then frozen) ----------
ptm        = {}            # "CHAIN:RESID": CODE (KCX SEP TPO PTR ...)
ptm_charge = {}                  # "CHAIN:RESID": int (overrides default charge)
# --- non-protein (HETATM) charge — NEVER assumed; declare it yourself --------------
# Per-residue formal charges (summed over INSTANCES, so two ZN at +2 -> +4), OR a
# single total via nonprotein_charge. Reported as NET_THEOZYME_NONPROTEIN_CHARGE.
# OPAA di-Zn site: 2x Zn(II) +2 + bridging OHX -1 + SUB 0  ->  +3
protonate_ligands = False      # stub (HETATM assumed already protonated)
ligand_charges    = {"ZN": 2, "OHX": -1, "SUB": 0}   # "RESNAME": charge per non-water HETATM
nonprotein_charge = None        # int OVERRIDES the dict with one total; None = use dict
# --- protomer tuning (only when protomers > 1) ---------------------
protomer_min_prob     = 0.15
protomer_max_variable = 12
couples = []                   # [("CHAIN:RESID","CHAIN:RESID"), ...] vary in lockstep
# --- optional MLFF relax of ONLY the new H (CPU, charge-free) -------
relax_h        = True
relax_h_model  = "mace-off-small"   # auto-falls back to mace-mp if metals present
relax_h_fmax, relax_h_steps, relax_h_device = 0.05, 200, "cpu"
# --- misc -----------------------------------------------------------
skip_propka          = False
keep_input_hydrogens = False
output_info_file     = None    # e.g. f"{PROTOMERS_DIR}protonation_info.json"
log_level            = "DEBUG"
# --- geometry cutoffs (advanced; defaults are sensible) ------------
bond_cutoff, metal_coord_cutoff, hbond_cutoff = 1.8, 2.8, 3.5
clash_cutoff, h_clash_cutoff, disulfide_cutoff = 2.0, 1.5, 2.5

### SANITY CHECKS ###
if not Path(input_pdb).is_file():
    raise FileNotFoundError(f"input_pdb not found: {input_pdb}")
Path(output_pdb).parent.mkdir(parents=True, exist_ok=True)

### BUILD THE COMMAND ###
cmd  = [*APPTAINER(CONTAINER, gpu=False), "python", PROTONATOR, "--input-pdb", input_pdb]
if output_pdb:                 cmd += ["--output-pdb", output_pdb]
cmd += ["--pH", str(pH), "--protomers", str(protomers), "--n-cap", n_cap, "--c-cap", c_cap]
for k, v in n_cap_overrides.items(): cmd += ["--n-cap-override", f"{k}={v}"]
for k, v in c_cap_overrides.items(): cmd += ["--c-cap-override", f"{k}={v}"]
for k, v in set_overrides.items():   cmd += ["--set", f"{k}={v}"]
for k, v in ptm.items():             cmd += ["--ptm", f"{k}={v}"]
for k, v in ptm_charge.items():      cmd += ["--ptm-charge", f"{k}={v}"]
for k, v in ligand_charges.items():  cmd += ["--ligand-charge", f"{k}={v}"]
if nonprotein_charge is not None:    cmd += ["--nonprotein-charge", str(nonprotein_charge)]
if protonate_ligands: cmd += ["--protonate-ligands"]
if protomers > 1:
    cmd += ["--protomer-min-prob", str(protomer_min_prob), "--protomer-max-variable", str(protomer_max_variable)]
    for a, b in couples: cmd += ["--couple", f"{a},{b}"]
if relax_h:
    cmd += ["--relax-h", "--relax-h-model", relax_h_model, "--relax-h-fmax", str(relax_h_fmax),
            "--relax-h-steps", str(relax_h_steps), "--relax-h-device", relax_h_device]
if skip_propka:          cmd += ["--skip-propka"]
if keep_input_hydrogens: cmd += ["--keep-input-hydrogens"]
if output_info_file:     cmd += ["--output-info-file", output_info_file]
cmd += ["--bond-cutoff", str(bond_cutoff), "--metal-coord-cutoff", str(metal_coord_cutoff),
        "--hbond-cutoff", str(hbond_cutoff), "--clash-cutoff", str(clash_cutoff),
        "--h-clash-cutoff", str(h_clash_cutoff), "--disulfide-cutoff", str(disulfide_cutoff)]
cmd += ["--log-level", log_level]

### PRINT COMMAND ###
if print_commands:
    print("### CONSTRUCTED COMMAND (copy-paste into terminal) ###")
    print(" ".join(str(x) for x in cmd))
    _out = output_pdb if output_pdb else input_pdb.rsplit(".", 1)[0] + "_protonated.pdb"
    if protomers > 1:
        print(f"\n# Output: {_out.rsplit('.pdb',1)[0]}_protomer1..{protomers}.pdb")
    else:
        print(f"\n# Output: {_out}")


### CONSTRUCTED COMMAND (copy-paste into terminal) ###
apptainer exec --bind /home --bind /net /net/software/containers/users/woodbuse/quantum_chem/quantum_chem-20260604.sif python /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/quantum_engine/prep/protonator.py --input-pdb /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/theozyme/input_theozyme/opaa_3l7g_optimal_maximal_theozyme_pxn_unprotonated.pdb --output-pdb /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/protomers/opaa_3l7g_optimal_maximal_theozyme_pxn.pdb --pH 7.5 --protomers 1 --n-cap nh2 --c-cap cho --ligand-charge ZN=2 --ligand-charge OHX=-1 --ligand-charge SUB=0 --relax-h --relax-h-model mace-off-small --relax-h-fmax 0.05 --relax-h-steps 200 --relax-h-device cpu --bond-cutoff 1.8 --metal-coord-cutoff 2.8 --hbond-cutoff 3.5 --clash-cutoff 2.0 --h-clash-cutoff 1.5 --disulfide-cutoff 2.5 --log-level DEBUG

# Output: /home/woodbuse/codeb

# **STEP 1: Monitor Active Site (bond / metal coordination)**

In [33]:
##################################################################
###          MONITOR ACTIVE SITE   (cowboy-qc monitor)              ###
##################################################################
# Non-constraining sanity report: measured key bonds + auto-detected metal
# coordination shells. Confirm the protonated geometry before spending GPU time.
# Instant CPU — this cell PRINTS the command (copy-paste; no sbatch).

print_commands = True

### INPUTS ###
input_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"

### OUTPUTS ###
out_dir = MONITOR_DIR

### MONITOR PARAMETERS ###
# Atom pairs to measure. Each atom is a descriptor — RESNAME-ATOM (OHX-O3),
# <Chain><ResNo>-ATOM (A519-ZN), <RESNAME><ResNo>-ATOM (ZN519-ZN), CHAIN:RESID:ATOM
# (A:519:ZN) — or a 0-based index. (`cowboy-qc monitor --bond` resolves these.)
monitor_bond_pairs = [("OHX-O3", "SUB-P1"),    # forming: hydroxide O -> phosphorus
                      ("SUB-P1", "SUB-O7")]    # breaking: phosphorus -> leaving-group O
report_metals      = True      # auto-detect metals + their coordination shells

### CONSTANTS ###
CONTAINER = MAIN_SIF

### BUILD THE COMMAND ###
cmd = [*APPTAINER(CONTAINER, gpu=False), *CLI, "monitor", input_pdb, "--outdir", out_dir]
for i, j in monitor_bond_pairs: cmd += ["--bond", f"{i},{j}"]
if report_metals: cmd += ["--metals"]

### PRINT COMMAND ###
if print_commands:
    print("### CONSTRUCTED COMMAND (copy-paste into terminal) ###")
    print(" ".join(str(x) for x in cmd))
    print(f"\n# JSON report → {out_dir}")


### CONSTRUCTED COMMAND (copy-paste into terminal) ###
apptainer exec --bind /home --bind /net /net/software/containers/users/woodbuse/quantum_chem/quantum_chem-20260604.sif env PYTHONPATH=/home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/ python -m quantum_engine.cli monitor /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/protomers/opaa_3l7g_optimal_maximal_theozyme_pxn.pdb --outdir /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/monitor/ --bond OHX-O3,SUB-P1 --bond SUB-P1,SUB-O7 --metals

# JSON report → /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/monitor/


# **STEP 2: Reaction Spec (clean variables → auto-generated YAML)**

In [35]:
##################################################################
###     REACTION SPEC   (define the chemistry, autogen YAML)   ###
##################################################################
# You set clean Python variables here; the cell BUILDS the ReactionSpec YAML for you (there is no hardcoded YAML blob to edit). The spec declares WHAT reacts:
# forming/breaking bonds, the reactive atoms (imag-mode overlap set), and an optional 1-D collective variable (cv) used by scans + the reactant-only entry.
#
# ATOM TOKENS (every cowboy-qc command that takes a PDB resolves these): RESNAME-ATOM (OHX-O3, SUB-P1), <Chain><ResNo>-ATOM (A519-ZN), <RESNAME><ResNo>-ATOM
#              (ZN519-ZN), CHAIN:RESID:ATOM (A:169:NZ), serial:N (1-based), 0:N, or a 0-based index. A token must be UNIQUE — ambiguous ones error with candidates.
# IMPORTANT: charge & multiplicity are NOT read from this YAML — always pass --charge/--multiplicity on every cowboy-qc command (the YAML keys would be silently ignored).

print_commands = True

### INPUTS ###
struct_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"        # used only to resolve atom tokens during validation

### OUTPUTS ###
spec_path = f"{REACTION_SPEC_DIR}reaction_spec.yaml"

### REACTION DEFINITION ###
# (nucleophile, electrophile) pairs that FORM; (atomA, atomB) pairs that BREAK.
forming_bonds  = [('OHX-O3', 'SUB-P1')]     # e.g. O_nuc -> P
breaking_bonds = [('SUB-P1', 'SUB-O7')]     # e.g. P -> O_leaving
reactive_atoms = ['OHX-O3', 'SUB-P1', 'SUB-O7']   # atoms on the imaginary mode

# Optional 1-D collective variable (bond_difference: s = d(a,b) - d(a,c) over EXACTLY 3
# atoms [a=center, b=breaking-partner, c=forming-partner]); set cv_kind=None to omit.
cv_kind  = "bond_difference"
cv_atoms = ['SUB-P1', 'SUB-O7', 'OHX-O3']   # [center, breaking, forming]
# Optional reactant->product atom map for double-ended methods ({} = identical ordering).
atom_map = {}

### CONSTANTS ###
CONTAINER = MAIN_SIF

### GENERATE YAML ###
def _tok(t):  return str(t)
def _bond(b): return f"  - [{_tok(b[0])}, {_tok(b[1])}]"
_lines = ["# auto-generated ReactionSpec (charge/spin live on the CLI, not here)"]
_lines += ["forming_bonds:"]  + [_bond(b) for b in forming_bonds]
_lines += ["breaking_bonds:"] + [_bond(b) for b in breaking_bonds]
_lines += ["reactive_atoms:"] + [f"  - {_tok(a)}" for a in reactive_atoms]
if cv_kind:
    _lines += ["cv:", f"  kind: {cv_kind}", f"  atoms: [{', '.join(_tok(a) for a in cv_atoms)}]"]
if atom_map:
    _lines += ["atom_map:"] + [f"  {k}: {v}" for k, v in atom_map.items()]
spec_yaml = "\n".join(_lines) + "\n"
Path(spec_path).parent.mkdir(parents=True, exist_ok=True)
Path(spec_path).write_text(spec_yaml)
print(f"# wrote {spec_path}\n"); print(spec_yaml)

### BUILD + PRINT VALIDATION COMMAND ###
cmd = [*APPTAINER(CONTAINER, gpu=False), *CLI, "reaction-spec", spec_path, "--structure", struct_pdb]
if print_commands:
    print("### VALIDATE (copy-paste into terminal) ###")
    print(" ".join(str(x) for x in cmd))

# wrote /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/reaction_spec/reaction_spec.yaml

# auto-generated ReactionSpec (charge/spin live on the CLI, not here)
forming_bonds:
  - [OHX-O3, SUB-P1]
breaking_bonds:
  - [SUB-P1, SUB-O7]
reactive_atoms:
  - OHX-O3
  - SUB-P1
  - SUB-O7
cv:
  kind: bond_difference
  atoms: [SUB-P1, SUB-O7, OHX-O3]

### VALIDATE (copy-paste into terminal) ###
apptainer exec --bind /home --bind /net /net/software/containers/users/woodbuse/quantum_chem/quantum_chem-20260604.sif env PYTHONPATH=/home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/ python -m quantum_engine.cli reaction-spec /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/reaction_spec/reaction_spec.yaml --structure /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/protomers/opaa_3l7g_optimal_maximal_theozyme_pxn.pdb


# **STEP 3: Minimize / Relax a Geometry (`cowboy-qc opt`)**

In [37]:
##################################################################
###     MINIMIZE / RELAX   (cowboy-qc opt; constrained or not)       ###
##################################################################
# Relax ANY input geometry: a reactant, a product, a TS-region pose, or the whole
# protonated cluster. The constraint regime is the key knob:
#   * unconstrained    (fix_preset='none')      -> a true minimum (final R / P endpoints)
#   * CA-frozen        (fix_preset='ca-only')   -> scaffold the backbone, let chemistry breathe
#   * bond-pinned      (fix_bonds=[...])        -> hold the forming/breaking distance(s) while everything else relaxes (constrained TS-region min)
# fix_bond / restrain_bond atoms accept any token: descriptor (OHX-O3, A519-ZN), serial:N, CHAIN:RESID:ATOM, or a 0-based index. (Trailing R0/K stays numeric.)

### INPUTS ###
input_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"        # the geometry to relax (reactant / product / ts-region / cluster)

### OUTPUTS ###
out_dir     = f"{RELAX_MINIMIZE_DIR}relax/"
relaxed_pdb = f"{out_dir}relaxed.pdb"

### ENERGY MODEL (plug-and-play: change `model`/`head`; container_for() picks the sif) ###
# Charged METAL active sites (e.g. di-Zn) — use a CHARGE-AWARE model:
#   mace-polar-m : polarizable + long-range electrostatics — IDEAL for a charged metal pocket (DEFAULT; baked into MAIN_SIF, loads in-process). sizes -s|-m|-l
#   mace-omol    : wB97M-V/OMol25, charge-aware, highest accuracy (large GPU: A6000/H200)
#   mace-mh-1    : multi-head foundation model -> set head='omol' for the OMol25 head
#   orb-mol-conservative : Orbital-Materials, charge/spin-aware, Zn-capable (true gradients)
#   uma-m-1p1 / esen-sm-conserving / allscaip-md-conserving : FairChem (route to UMA_SIF)
# Organic-only (NO metals): mace-off-* (wB97M organic) ; aimnet2-rxn (CHON, TS-tuned).
# AVOID GFN2-xTB on metals (not charge-aware there). --head applies to MACE multi-head only.
model        = 'mace-polar-m'
head         = None            # MACE multi-head only (e.g. 'omol' for mace-mh-1); None for polar/omol
device       = "cuda"
charge       = 0               # FULL-cluster net charge (CLI-only; the spec YAML ignores charge/multiplicity)
multiplicity = 1               # spin MULTIPLICITY M=2S+1 (1=singlet/no radicals, 2=doublet, 3=triplet) # NOTE: the CLI flag is --multiplicity. (--spin takes S and converts to 2S+1.)

### CONSTRAINTS ###
fix_preset     = "ca-only"      # 'none' | 'ca-only' | 'backbone' | 'backbone-water'
extra_fix      = []             # extra select specs, e.g. ['residue HOH', 'chain B', 'resid 169']
extra_free     = []             # subtract from the preset, e.g. ['atoms ZN1 ZN2']
fix_bonds      = []             # hard-pin: [[A, B]] or [[A, B, R0]] (atom tokens)  e.g. [['SUB-P1', 'OHX-O3']]
restrain_bonds = []             # harmonic: [[i, j, K, R0]]  (0-based; K in eV/A^2)

### TS-GUESS AUTO-PIN (pull forming/breaking bonds from the STEP-2 reaction spec) ###
# A TS-region pose is a SADDLE: a plain minimizer rolls downhill OFF it and collapses to the
# reactant or product. Set is_ts_guess=True to auto-pin the reaction-spec's forming+breaking
# bonds (at their current lengths) so only the scaffold/H-bonds/waters relax. Pins in `fix_bonds`
# above are ADDED on top; set is_ts_guess=False for a reactant/product/cluster (a real minimum).
is_ts_guess        = True
reaction_spec_yaml = f"{REACTION_SPEC_DIR}reaction_spec.yaml"   # STEP 2 output (forming/breaking bonds)

### OPT PARAMETERS ###
optimizer = "lbfgs"             # lbfgs | bfgs | fire
fmax      = 0.05                # eV/A (0.05 is fine for MLFFs)
max_steps = 500

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_relax"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(input_pdb).is_file():
    raise FileNotFoundError(f"input_pdb not found: {input_pdb}")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd  = qcb_cmd(model, "opt", input_pdb, "--model", model, "--charge", charge, "--multiplicity", multiplicity, "--device", device)
if head: cmd += ["--head", head]
cmd += ["--fix-preset", fix_preset]
for s in extra_fix:  cmd += ["--fix", s]
for s in extra_free: cmd += ["--free", s]
# is_ts_guess: pin the STEP-2 reaction spec's forming+breaking bonds (held at current length)
auto_fix_bonds = []
if is_ts_guess:
    import yaml
    if not Path(reaction_spec_yaml).is_file():
        raise FileNotFoundError(f"is_ts_guess=True needs the STEP-2 reaction spec: {reaction_spec_yaml}")
    _rs = yaml.safe_load(Path(reaction_spec_yaml).read_text()) or {}
    auto_fix_bonds = [list(b) for b in (_rs.get("forming_bonds") or [])] + \
                     [list(b) for b in (_rs.get("breaking_bonds") or [])]
    print(f"# is_ts_guess=True -> auto-pinned {len(auto_fix_bonds)} reaction bond(s): {auto_fix_bonds}")
for b in (auto_fix_bonds + list(fix_bonds)): cmd += ["--fix-bond", *map(str, b)]
for b in restrain_bonds: cmd += ["--restrain-bond", *map(str, b)]
cmd += ["--optimizer", optimizer, "--fmax", fmax, "--max-steps", max_steps,
        "--outdir", out_dir, "--output-pdb", relaxed_pdb]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output: {relaxed_pdb}")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '02:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# is_ts_guess=True -> auto-pinned 2 reaction bond(s): [['OHX-O3', 'SUB-P1'], ['SUB-P1', 'SUB-O7']]
# 1 command(s) → /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/cmds/opaa_theozyme_relax

apptainer exec --nv --bind /home --bind /net /net/software/containers/users/woodbuse/quantum_chem/quantum_chem-20260604.sif env PYTHONPATH=/home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/ python -m quantum_engine.cli opt /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/protomers/opaa_3l7g_optimal_maximal_theozyme_pxn.pdb --model mace-polar-m --charge 0 --multiplicity 1 --device cuda --fix-preset ca-only --fix-bond OHX-O3 SUB-P1 --fix-bond SUB-P1 SUB-O7 --optimizer lbfgs --fmax 0.05 --max-steps 500 --outdir /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_theozyme/output/relax_minimize/relax/ --output-pdb /home/woodbuse/codebase_projects/quantum_cowboy_biochemistry/notebooks/opaa_th

# **STEP 4: 1-D Relaxed Scan → Reactant, Product & TS Guess (`cowboy-qc scan`)**

A 1-D *relaxed* scan slides one reaction coordinate and re-minimizes everything else at each
step. In one shot it yields the **reactant** (one end), the **product** (other end), and a
**TS guess** (highest-energy frame) that feeds `refine-ts`.

**Two coordinate choices — set `scan_kind`:**

| `scan_kind` | coordinate | when to use |
|---|---|---|
| `"bond"` | one forming/breaking bond distance `d(i,j)` | a single bond dominates the step |
| `"bond-difference"` | the More-O'Ferrall–Jencks CV `s = d(center,breaking) − d(center,forming)` | **SN2-like** steps — drives both bonds antisymmetrically and does **not** pre-bias concerted vs stepwise *(recommended here)* |

**Direction matters** — the scan always runs **reactant → product**, so `frame[0]` is the reactant
and `frame[-1]` the product:
- `"bond"` on the *forming* bond: **long** (nucleophile far ⇒ reactant) → **short** (bonded ⇒ product).
- `"bond-difference"`: **s ≪ 0** (reactant) → **s ≫ 0** (product).

> ⚠️ **Your monitor showed the active site already near the TS** (O3–P ≈ 1.97 Å bonded, P–O7
> stretched). So push the **reactant** end far enough (O3–P ≳ 3.2 Å, or `s ≈ −2.5`) to reach a
> clean reactant basin, and don't be surprised if the TS guess lands near `frame[0]`. If a
> **pentacoordinate phosphorane intermediate** is real you'll see a dip mid-scan — the optional
> **2-D scan below** resolves concerted vs stepwise.

**Troubleshooting**
- *Product never forms / reactant never separates* → widen `scan_start`/`scan_end`.
- *Jagged profile / hysteresis* → more `scan_n_steps`, lower `scan_fmax`, or use `"bond-difference"`.
- *A bond snaps or the cluster distorts* → tighten `fix_preset` (e.g. `"backbone"`) so only the coordinate moves.
- *Endpoints look swapped* → re-check the direction note above (`frame[0]` must be the reactant).

In [ ]:
##################################################################
###     1-D RELAXED SCAN   (cowboy-qc scan; reactant -> product) ###
##################################################################
# Slides the reaction coordinate and relaxes everything else at each step ->
# reactant (frame 0), product (last frame), TS guess (highest-E frame -> refine-ts).
# Atom tokens are descriptors (OHX-O3), serial:N, CHAIN:RESID:ATOM, or 0-based indices.

### INPUTS ###
relaxed_pdb = f"{RELAX_MINIMIZE_DIR}relax/relaxed.pdb"   # usually the CA-frozen relaxed cluster

### OUTPUTS ###
out_dir           = f"{SCAN_DIR}scan/"
ts_guess_pdb      = f"{out_dir}ts_guess.pdb"        # max-energy frame -> Step refine-ts
reactant_scan_pdb = f"{out_dir}reactant_scan.pdb"  # frame 0   (approx reactant) -> minimize next
product_scan_pdb  = f"{out_dir}product_scan.pdb"   # last frame (approx product) -> minimize next

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"   # default mace-polar-m (see relax cell's menu)
charge, multiplicity = 0, 1                 # net charge (CLI-only); multiplicity M=2S+1 (1=singlet)

### CONSTRAINTS ###
fix_preset = "ca-only"          # the scanned coordinate is auto-pinned ON TOP of this preset

### REACTION-COORDINATE ATOMS (descriptors) ###
center   = 'SUB-P1'     # shared atom = the electrophilic centre (P)
forming  = 'OHX-O3'   # FORMING-bond partner (nucleophile -> centre)
breaking = 'SUB-O7'    # BREAKING-bond partner (centre -> leaving group)

### SCAN PARAMETERS ###
scan_kind    = "bond-difference"   # "bond" (one forming bond) | "bond-difference" (MOJ CV; recommended for SN2)
scan_n_steps = 16
scan_fmax    = 0.05
# Direction is ALWAYS reactant -> product, so frame[0]=reactant and frame[-1]=product:
if scan_kind == "bond":
    scan_coord, scan_indices = "bond", [forming, center]   # the forming bond, e.g. O_nuc..P
    scan_start, scan_end = 3.2, 1.6        # Angstrom: reactant (nuc far) -> product (nuc bonded)
    traj_name = "scan-trajectory.xyz"
elif scan_kind == "bond-difference":
    scan_coord, scan_indices = "bond-difference", [center, breaking, forming]
    scan_start, scan_end = -2.5, 2.5       # CV s (Angstrom): reactant (s<0) -> product (s>0)
    traj_name = "scan-bonddiff-trajectory.xyz"
else:
    raise ValueError("scan_kind must be 'bond' or 'bond-difference'")

### CONSTANTS ###
template_pdb = relaxed_pdb      # residue-annotation template for the extracted PDB

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_scan"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(relaxed_pdb).is_file():
    raise FileNotFoundError(f"relaxed_pdb not found: {relaxed_pdb}  (run the relax step first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
# 1) the scan; 2) a helper that pulls reactant/product/TS-guess frames out as PDBs.
commands = []
cmd_scan  = qcb_cmd(model, "scan", relaxed_pdb, "--model", model, "--charge", charge,
                    "--multiplicity", multiplicity, "--device", device, "--fix-preset", fix_preset,
                    "--coord", scan_coord, "--indices", *scan_indices,
                    "--start", scan_start, "--end", scan_end, "--n-steps", scan_n_steps,
                    "--fmax", scan_fmax, "--outdir", out_dir)
extract_py = f"{out_dir}extract_frames.py"
Path(extract_py).write_text(textwrap.dedent(f"""\
    import ase.io as io, json, glob
    from quantum_engine.io import load_structure, write_pdb
    OUT      = r'{out_dir}'
    TRAJ     = r'{out_dir}{traj_name}'
    TEMPLATE = r'{template_pdb}'
    CHARGE   = {charge}
    frames = io.read(TRAJ, index=':')   # frame 0 = reactant, last = product
    energy = lambda a: a.info['energy_eV'] if 'energy_eV' in a.info else a.get_potential_energy()
    # TS-guess frame = highest INTERIOR maximum (the scan summary picks it; endpoints are basins).
    sums = sorted(glob.glob(OUT + '*-summary.json'))
    if sums:
        s = json.loads(open(sums[0]).read()); i = int(s['ts_guess_idx']); barrierless = s.get('barrierless', False)
    else:
        es = [energy(a) for a in frames]
        interior = [k for k in range(1, len(es) - 1) if es[k] >= es[k - 1] and es[k] >= es[k + 1]]
        i = max(interior, key=lambda k: es[k]) if interior else max(range(len(es)), key=lambda k: es[k])
        barrierless = not interior
    _, bt, _ = load_structure(TEMPLATE)
    write_pdb(frames[0],  bt, r'{reactant_scan_pdb}', total_charge=CHARGE)
    write_pdb(frames[-1], bt, r'{product_scan_pdb}',  total_charge=CHARGE)
    write_pdb(frames[i],  bt, r'{ts_guess_pdb}',      total_charge=CHARGE)
    print('TS guess = frame %d/%d ; reactant=frame 0 ; product=frame %d' % (i, len(frames) - 1, len(frames) - 1))
    if barrierless:
        print('# WARNING: no interior barrier (monotonic) -> TS guess is an endpoint; widen/shift the scan range.')
"""))
cmd_extract = [*APPTAINER(container_for(model)), "python", extract_py]
commands.append(" ".join(str(x) for x in cmd_scan))
commands.append(" ".join(str(x) for x in cmd_extract))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# scan_kind={scan_kind} ; {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {out_dir}{traj_name}, scan*-summary.json, scan*.png")
print(f"#          TS guess: {ts_guess_pdb} ; endpoints: {reactant_scan_pdb} (reactant), {product_scan_pdb} (product)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '04:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 2
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


## *(optional)* 2-D Relaxed Scan — concerted vs stepwise diagnostic (`cowboy-qc scan2d`)

Drives the **forming** (center–forming) and **breaking** (center–breaking) bonds *independently*
on a grid around a TS guess, relaxing everything else. Reading the 2-D energy map:
- a single **diagonal ridge** ⇒ **concerted** (one TS; both bonds change together);
- a distinct **off-diagonal basin** ⇒ **stepwise** via a **pentacoordinate phosphorane** intermediate.

Diagnostic only (no endpoints) and ~grid² relaxations, so it's pricier — run it when you suspect a
stepwise mechanism (as the near-TS geometry here hints); skip it for a routine concerted step.

In [ ]:
##################################################################
###  (OPTIONAL) 2-D RELAXED SCAN   (cowboy-qc scan2d; diagnostic) ###
##################################################################
# Grid scan of the FORMING (center-forming) x BREAKING (center-breaking) bonds around a
# TS guess. Off-diagonal basin => stepwise (pentacoordinate intermediate); diagonal ridge
# => concerted. PRINTS the command (copy-paste / sbatch). ~grid^2 relaxations.

print_commands = True

### INPUTS ###
ts_guess_pdb = f"{SCAN_DIR}scan/ts_guess.pdb"   # from the 1-D scan above (or any TS-like geometry)

### OUTPUTS ###
out_dir_2d = f"{SCAN_DIR}scan2d/"

### COORDINATE BONDS (descriptors) + ENERGY MODEL ###
center, forming, breaking = 'SUB-P1', 'OHX-O3', 'SUB-O7'
model, head, device  = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### SCAN2D PARAMETERS ###
grid    = "5x5"     # n_a x n_b relaxed grid points (cost ~ n_a*n_b)
delta_d = 0.20      # Angstrom step per bond, out from the TS guess

### SANITY + COMMAND ###
Path(out_dir_2d).mkdir(parents=True, exist_ok=True)
cmd_2d = qcb_cmd(model, "scan2d", "--input", ts_guess_pdb, "--ts-guess", ts_guess_pdb,
                 "--bond-a", f"{forming},{center}", "--bond-b", f"{center},{breaking}",
                 "--grid", grid, "--delta-d", delta_d, "--charge", charge,
                 "--multiplicity", multiplicity, "--device", device, "--outdir", out_dir_2d)
if head: cmd_2d += ["--head", head]
if print_commands:
    print("### (optional) 2-D scan command (copy-paste / sbatch) ###")
    print(" ".join(str(x) for x in cmd_2d))
    print(f"\n# Output: {out_dir_2d}  (energy grid + plot; off-diagonal basin = stepwise/pentacoordinate)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '04:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 5: Analysis — scan profile + TS guess**

In [ ]:
##################################################################
###  ANALYSIS: 1-D SCAN PROFILE + TS GUESS  (run after the scan) ###
##################################################################
# Reads the scan summary + trajectory, plots the profile, and reports the forward
# barrier + TS-guess frame. The TS guess is the highest INTERIOR maximum (the scan
# engine picks it — endpoints are basins in a relaxed scan). Auto-finds the outputs
# by glob, so it works for ANY scan_kind (scan-* or scan-bonddiff-*).

### INPUTS ###
scan_dir = f"{SCAN_DIR}scan/"

### ANALYSIS ###
import json, glob
from ase.io import read as _read
_trajs = sorted(glob.glob(str(Path(scan_dir) / "*-trajectory.xyz")))
_sums  = sorted(glob.glob(str(Path(scan_dir) / "*-summary.json")))
if not _trajs:
    print(f"# no *-trajectory.xyz yet at {scan_dir} — run the scan step first.")
else:
    frames = _read(_trajs[0], index=":")
    e   = [a.info["energy_eV"] if "energy_eV" in a.info else 0.0 for a in frames]
    crd = [a.info.get("scan_value", a.info.get("s_target", i)) for i, a in enumerate(frames)]
    e0  = min(e); ek = [(x - e0) * 23.0605 for x in e]      # eV -> kcal/mol, relative
    s   = json.loads(Path(_sums[0]).read_text()) if _sums else {}
    ts  = int(s.get("ts_guess_idx", max(range(len(e)), key=lambda k: e[k])))   # interior max (engine)
    print(f"scan trajectory         : {Path(_trajs[0]).name}  ({len(frames)} frames)")
    print(f"forward barrier         : {s.get('barrier_kcal', ek[ts]):.2f} kcal/mol")
    print(f"TS guess                : frame {ts} of {len(frames) - 1}  (coord {crd[ts]:+.3f})")
    print(f"reactant=frame 0 (coord {crd[0]:+.3f})  |  product=frame {len(frames) - 1} (coord {crd[-1]:+.3f})")
    if s.get("barrierless"):
        print("# WARNING: no interior barrier (monotonic) — TS guess is an endpoint; widen/shift the range.")
    try:
        fig, ax = plt.subplots(figsize=(5, 3.2))
        ax.plot(crd, ek, "o-", color=nb.good_teal)
        ax.axvline(crd[ts], ls="--", color=nb.good_red, label="TS guess")
        ax.set_xlabel("scan coordinate"); ax.set_ylabel("rel. energy (kcal/mol)")
        ax.set_title("1-D relaxed scan profile"); ax.legend(); plt.tight_layout(); plt.show()
    except Exception as _ex:
        print(f"# (plot skipped: {_ex})")


# **STEP 6: Minimize the Reactant & Product Endpoints (`cowboy-qc opt`)**

In [ ]:
##################################################################
###  MINIMIZE ENDPOINTS  (scan ends -> TRUE reactant / product) ###
##################################################################
# WHY: the barrier is E(TS) - E(reactant_min); you must compare two TRUE stationary
# points. The scan's first/last frames are constraint-biased approximations, so relax
# each to a real minimum. Use the SAME CA-frozen scaffold + model/charge/spin as the TS
# so the energies are comparable. Reactive bonds are FREE here (no --fix-bond) — these are
# basins, not the TS. These two minima are also what verify-irc-like should reproduce.

### INPUTS ###
reactant_scan_pdb = f"{SCAN_DIR}scan/reactant_scan.pdb"   # first scan frame
product_scan_pdb  = f"{SCAN_DIR}scan/product_scan.pdb"    # last  scan frame

### OUTPUTS ###
out_dir = f"{RELAX_MINIMIZE_DIR}endpoints/"
# Each endpoint gets its OWN subdir so their opt-summary.json energies don't collide
# (the barrier-analysis cell reads both: barrier = E(TS) - E(reactant_min)).
R_min   = f"{out_dir}reactant/reactant_min.pdb"
P_min   = f"{out_dir}product/product_min.pdb"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"   # default mace-polar-m (see relax cell's menu)
charge, multiplicity = 0, 1                 # net charge (CLI-only); multiplicity M=2S+1 (1=singlet)

### CONSTRAINTS ###
fix_preset = "ca-only"          # same scaffold as the TS; reactive bonds FREE (no pin)

### OPT PARAMETERS ###
optimizer, fmax, max_steps = "lbfgs", 0.03, 500   # tighter fmax for clean basins

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_min_endpoints"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
for p in (reactant_scan_pdb, product_scan_pdb):
    if not Path(p).is_file():
        raise FileNotFoundError(f"scan endpoint not found: {p}  (run the scan step first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
for _src, _dst in [(reactant_scan_pdb, R_min), (product_scan_pdb, P_min)]:
    _sub = str(Path(_dst).parent)
    cmd = qcb_cmd(model, "opt", _src, "--model", model, "--charge", charge, "--multiplicity", multiplicity,
                  "--device", device, "--fix-preset", fix_preset, "--optimizer", optimizer,
                  "--fmax", fmax, "--max-steps", max_steps, "--outdir", _sub, "--output-pdb", _dst)
    if head: cmd += ["--head", head]
    commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {R_min} , {P_min}  (barrier = E(TS) - E(reactant_min))")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '03:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 2
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 7: *(optional, more rigorous than the 1-D scan)* Path search — CI-NEB / GSM / FSM (one cell, pick `path_method`)**

In [ ]:
##################################################################
###  DOUBLE-ENDED PATH SEARCH  (CI-NEB | GSM | FSM — selectable)  ###
##################################################################
# ONE cell, plug-and-play method. Between a reactant and product (the MINIMIZED
# endpoints), find a TS guess. Set `path_method`:
#   'ci-neb' : climbing-image NEB (cowboy-qc neb) — robust, geodesic interp, honours --fix-preset
#   'gsm'    : Growing String (cowboy-qc gsm --method gsm) — cheaper; honours --fix-preset (pysisyphus freeze_atoms)
#   'fsm'    : Freezing String (cowboy-qc gsm --method fsm) — cheapest; honours --fix-preset (pysisyphus freeze_atoms)
# RECOMMENDED for LARGE, flexible clusters with a DISSOCIATIVE step: CI-NEB — its geodesic interpolation routes the
# path AROUND repulsive walls. GSM/FSM grow nodes by pysisyphus internal-coord interpolation (no geodesic), which can
# park a node on a high-energy wall and report a spurious 'barrier'; they suit smaller / gas-phase / concerted reactions.
# (AutoNEB / pyGSM / single-ended SE-GSM are also available via
#  `cowboy-qc ts-entry --path-method {autoneb|pygsm-de|gsm-se}`.)
# A 1-D scan can slice BESIDE the true saddle for an ASYNCHRONOUS step; a double-ended
# search relaxes all orthogonal DOFs, so it's a better guess there. Feed CI-NEB to
# refine-ts via --from-neb; for GSM/FSM use the emitted TS-guess structure.

### INPUTS ###
reactant_min = f"{RELAX_MINIMIZE_DIR}endpoints/reactant/reactant_min.pdb"   # or any reactant basin
product_min  = f"{RELAX_MINIMIZE_DIR}endpoints/product/product_min.pdb"     # same atom order!

### OUTPUTS ###
path_method = "ci-neb"          # 'ci-neb' | 'gsm' | 'fsm'   <-- pick the method
out_dir     = f"{PATH_SEARCH_DIR}{path_method}/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### PATH PARAMETERS ###
n_images      = 17              # CI-NEB publication-tier (11 quicker); GSM/FSM ~15
interpolation = "geodesic"      # CI-NEB only — REQUIRED for dense/charged sites (never 'linear')
optimizer     = "fire"          # CI-NEB only
fix_preset    = "ca-only"       # honoured by CI-NEB AND GSM/FSM (GSM/FSM via pysisyphus freeze_atoms)
fmax          = 0.05

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_path_{path_method}"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
for p in (reactant_min, product_min):
    if not Path(p).is_file():
        raise FileNotFoundError(f"endpoint not found: {p}  (run min-endpoints first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
if path_method == "ci-neb":
    cmd = qcb_cmd(model, "neb", reactant_min, product_min, "--model", model, "--charge", charge,
                  "--multiplicity", multiplicity, "--device", device, "--fix-preset", fix_preset,
                  "--n-images", n_images, "--interpolation", interpolation, "--optimizer", optimizer,
                  "--outdir", out_dir)
    _feed = f"refine-ts: set from_neb = '{out_dir}'"
elif path_method in ("gsm", "fsm"):
    # GSM/FSM now honour --fix-preset via pysisyphus freeze_atoms (grown nodes inherit it via Geometry.copy()).
    cmd = qcb_cmd(model, "gsm", reactant_min, product_min, "--method", path_method, "--model", model,
                  "--charge", charge, "--multiplicity", multiplicity, "--device", device,
                  "--fix-preset", fix_preset, "--n-images", n_images, "--fmax", fmax,
                  "--outdir", out_dir)
    _feed = f"refine-ts: feed the TS-guess structure written under {out_dir}"
else:
    raise ValueError(f"path_method must be ci-neb|gsm|fsm, got {path_method!r}")
if head: cmd += ["--head", head]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) [{path_method}] → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output dir: {out_dir}  → {_feed}")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '04:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


### 📋 Path-search results (this run — OPAA di-Zn)

**CI-NEB ✅ (the right tool here)** — reactant_min → product_min, geodesic interp, ca-only: clean **monotonic single
barrier**, validation 4/4 PASS, **pentacoordinate TS at +8.12 kcal/mol** (O3–P 1.72 / P–O7 1.99 Å) → concerted SN2-at-P.
This climbing image is our TS guess.

**frozen-GSM ❌ (not suitable here)** — *identical endpoints*, yet reported a spurious **+17.84 kcal** 'barrier' that
refined into the cleaved-product basin (P–O7 4.97 Å). **Why:** GSM (pysisyphus `GrowingString`) grows nodes by
**internal-coordinate interpolation with no geodesic / clash-avoidance**, so a node parked on the repulsive
dissociative wall (P–O7 ≈ 2.8 Å). CI-NEB's **geodesic** interpolation routes around that, through the real
low-energy pentacoordinate MEP. Same endpoints, different node placement — not a bug, a method fit: use **CI-NEB**
for large flexible clusters with a dissociative step; GSM/FSM suit smaller / gas-phase / concerted reactions.


# **STEP 8: Refine to a Saddle (`cowboy-qc refine-ts`)**

In [ ]:
##################################################################
###  REFINE-TS  (saddle search + partial-Hessian acceptance)   ###
##################################################################
# The acceptance core: dimer/Sella/pysisyphus saddle search -> partial Hessian on the
# reactive atoms -> require exactly ONE imaginary mode (< cutoff) overlapping the reaction
# coordinate -> ts_refined.pdb. Input is EITHER a TS-guess PDB or a path-search dir
# (--from-neb). --reactive-atoms accept atom tokens: descriptor (OHX-O3), serial:N, index.

### INPUTS ###
ts_guess_pdb = f"{SCAN_DIR}scan/ts_guess.pdb"   # EITHER a guess PDB ...
from_neb     = None                               # ... OR a path-search dir, e.g. f"{PATH_SEARCH_DIR}neb/"
template_pdb = f"{PROTOMERS_DIR}opaa_3l7g_optimal_maximal_theozyme_pxn.pdb"                         # residue-annotation template (used with --from-neb)

### OUTPUTS ###
out_dir = f"{REFINE_TS_DIR}refine/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### CONSTRAINTS ###
fix_preset = "ca-only"          # kept during saddle + freq

### REFINE-TS PARAMETERS ###
reactive_atoms   = ['OHX-O3', 'SUB-P1', 'SUB-O7']   # atom tokens (nucleophile, center P, leaving group)
backend          = "auto"       # auto (sella->sella-internal->dimer) | dimer | sella | sella-internal | pysisyphus-rsprfo
saddle_fmax      = 0.02
saddle_max_steps = 500
imag_cm_cutoff   = -50.0        # imag mode must be MORE negative than this
imag_overlap     = 0.5          # >= this fraction of the mode on the reactive atoms
n_imag_expected  = 1            # first-order saddle

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_refine_ts"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if from_neb is None and not Path(ts_guess_pdb).is_file():
    raise FileNotFoundError(f"ts_guess_pdb not found: {ts_guess_pdb}  (or set from_neb=)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "refine-ts")
if from_neb: cmd += ["--from-neb", from_neb, "--template-pdb", template_pdb]
else:        cmd += [ts_guess_pdb]
cmd += ["--model", model, "--charge", charge, "--multiplicity", multiplicity, "--device", device,
        "--fix-preset", fix_preset, "--reactive-atoms", *map(str, reactive_atoms),
        "--backend", backend, "--saddle-fmax", saddle_fmax, "--saddle-max-steps", saddle_max_steps,
        "--imag-cm-cutoff", imag_cm_cutoff, "--imag-mode-overlap", imag_overlap,
        "--n-imag-expected", n_imag_expected, "--outdir", out_dir]
if head: cmd += ["--head", head]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {out_dir}ts_refined.pdb (validated TS), imag_mode.npy, summary.json (PASS/FAIL)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '04:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


### 📋 Refine-TS results (this run — OPAA di-Zn) — ⚠️ transition state NOT yet validated

CI-NEB gives a clean barrier (**~8 kcal/mol**, concerted SN2-at-P through a pentacoordinate region), but pinning a
**Hessian-verified saddle is failing** — a genuine difficulty on this large, flat, flexible MLFF surface:

- `--backend auto` (Sella-Cartesian) **collapses** the TS ~14 kcal/mol downhill into a basin (n_imag=0).
- `--backend dimer` (tangent-seeded) **diverges** — perturbs all atoms and tears the active site apart (hydroxide flies
  13 Å, E +325 kcal/mol); not a crash, but non-physical. `sella-internal` wandered 1h45m without converging.
- A **direct partial Hessian** on the CI-NEB climbing image (4 reactive atoms) → **0 imaginary modes** (lowest +203 cm⁻¹).

**Resolved:** the barrier region is a **flat ridge** (fine cliff-scan: ~1 kcal relaxed plateau; constrained NEB ~6–8). The reaction coordinate has no imaginary curvature at the pentacoordinate structures; the soft ‘imaginary’ modes (−55/−65 cm⁻¹) are spurious peripheral wobbles. **No clean first-order saddle at the MLFF level** — best TS structure `neb_n25/ts.pdb`; use DFT/QM/MM for a quantitative barrier.
which a small partial Hessian holds fixed and cannot realize. **Validation study (in flight):** a **45-atom Hessian**
over the whole reacting region to reveal a delocalized imaginary mode, plus **finer CI-NEBs (15/20/25 images)** to check
barrier / TS-geometry convergence.

**Codebase lesson (now documented + guarded):** never trust an `auto` refine that collapses below the NEB barrier; on
flat enzyme surfaces validate the climbing image directly (`freq`) or with a CV-constrained pin. See
`docs/ts_search_pitfalls_and_methods.md` and the auto `ts_sanity` warnings (TS≈endpoint / spurious-peak / collapse).


---
## **STEP 8b: TS refinement & validation — the *sharp-saddle* protocol**

When `refine-ts --backend auto` **collapses** (Sella-Cartesian minimises into a basin, `n_imag=0`) or
**diverges** (the dimer perturbs all atoms and tears the active site apart) — common on a large, flat enzyme
PES — the saddle is a **sharp feature** the standard optimizers can't pin. This is the fallback developed on
the OPAA di-Zn case (see `docs/ts_search_pitfalls_and_methods.md`). Three driver cells follow:

1. **NEB convergence study** — CI-NEB at several image counts. A real barrier converges; a *spurious* peak (a
   node on a repulsive wall) jumps around. Here n=11→8.1, **n=15→32 (spike)**, n=20→12.1, n=25→6.4 kcal/mol —
   the spike is caught automatically by the `ts_sanity` guard (it drops >8 kcal/mol to both neighbours).
2. **Fine cliff-scan** — a *narrow, dense* bond-difference scan across the barrier top to pinpoint the maximum.
   (The wide pipeline scan under-estimates because it relaxes orthogonal modes; a fine scan near the crest does not.)
3. **Reaction-region Hessian + mode-character check** — a partial Hessian over the **whole reacting moiety**
   (substrate + nucleophile + metals + first shell, NOT 3 atoms), then a check that the imaginary mode is the
   **reaction coordinate** — a small Hessian misses delocalized metalloenzyme modes, and a peripheral wobble is a false positive.


**Result (mace-polar-m) — honest bottom line:** concerted SN2-at-P, symmetric pentacoordinate (O3–P 1.85 / P–O7 1.83). The barrier region is a **FLAT RIDGE** — relaxed scan ~1 kcal/mol plateau, constrained NEB ~6–8 kcal/mol — with **NO well-defined first-order saddle**: the reaction coordinate (O3/P/O7) has near-zero/positive curvature, and the only ‘imaginary’ modes found (−55 / −65 cm⁻¹) are **spurious peripheral wobbles** (0 reactive-atom amplitude — the `ts_sanity` mode-character check catches them). A Hessian-verified TS and a precise barrier are **not obtainable at this MLFF level** (the PES is too flat here) — carry **`output/path_search/neb_n25/ts.pdb`** (symmetric pentacoordinate, ~6 kcal MLFF) to **DFT or QM/MM** for a quantitative barrier + saddle. Reaction exothermic ~14.5 kcal/mol.
barrier **~6–8 kcal/mol**, reaction exothermic ~14.5 kcal/mol. Best TS guess: `output/path_search/neb_n25/ts.pdb`
(Hessian-verification of the reaction-coordinate mode is the remaining step).


In [ ]:
### 8b-i  NEB CONVERGENCE STUDY  (barrier vs n_images; a real barrier converges, a spike is spurious) ###
reactant_min = f"{RELAX_MINIMIZE_DIR}endpoints/reactant/reactant_min.pdb"
product_min  = f"{RELAX_MINIMIZE_DIR}endpoints/product/product_min.pdb"
image_counts = [11, 15, 20, 25]
for N in image_counts:
    out = f"{PATH_SEARCH_DIR}neb_n{N}/"; Path(out).mkdir(parents=True, exist_ok=True)
    cmd = qcb_cmd('mace-polar-m', "neb", reactant_min, product_min, "--model", "mace-polar-m",
                  "--charge", 0, "--multiplicity", 1, "--device", "cuda", "--fix-preset", "ca-only",
                  "--n-images", N, "--interpolation", "geodesic", "--optimizer", "fire", "--outdir", out)
    print(" ".join(str(x) for x in cmd) + "\n")
# Submit as a 4-task array. Then compare profile.json['barrier_fwd_kcal'] across N; a value far above the
# others whose peak drops >8 kcal/mol to BOTH neighbours (profile['energy_rel_kcal']) is a spurious spike — discard.


In [ ]:
### 8b-ii  FINE CLIFF-SCAN  (dense bond-difference scan across the crest -> saddle estimate) ###
ts_guess = f"{PATH_SEARCH_DIR}neb_n25/ts.pdb"      # best-resolved climbing image (near the crest, s≈0)
out = f"{SCAN_DIR}cliff/"; Path(out).mkdir(parents=True, exist_ok=True)
cmd = qcb_cmd('mace-polar-m', "scan", ts_guess, "--model", "mace-polar-m", "--charge", 0,
              "--multiplicity", 1, "--device", "cuda", "--fix-preset", "ca-only",
              "--coord", "bond-difference", "--indices", "SUB-P1", "SUB-O7", "OHX-O3",
              "--start", -0.6, "--end", 0.6, "--n-steps", 13, "--fmax", 0.03, "--outdir", out)
print(" ".join(str(x) for x in cmd))
# The MAX-energy frame of {out}scan-bonddiff-trajectory.xyz is the saddle estimate -> write it to cliff_ts.pdb,
# then validate it with cell 8b-iii.


### 8b-iv  CROSS-MODEL BARRIER CHECK — is the flat barrier model-specific, or real?

Single-point energies of the **same** reactant / TS / product with **independent** charge-aware MLFFs
(run in the UMA sidecar; `output/model_compare/model_compare.py`):

| model | barrier(TS−R) | ΔE_rxn(P−R) | (kcal/mol) |
|---|---|---|---|
| mace-polar-m | ~6–8 (NEB) | −14.5 | reference |
| **UMA-M-1p1** | **5.8** | −13.4 | FairChem |
| **eSEN-sm-conserving** | **6.0** | −13.8 | FairChem |

**Three independent models AGREE on a ~6 kcal/mol barrier and ~−14 kcal exothermicity.** So the
flat/shallow barrier is **robust, not a mace-polar-m artifact** — strong evidence the chemical step is
**genuinely low-barrier** (consistent with an efficient di-Zn catalyst). Caveat: all three are MLFFs trained
on OMol25-class data, so a **DFT (or QM/MM)** check on `neb_n25/ts.pdb` remains the gold-standard confirmation,
ideally alongside an **uncatalyzed-reaction reference** to quantify the rate enhancement.


In [ ]:
### 8b-iii  REACTION-REGION HESSIAN + MODE-CHARACTER CHECK  (validate a TS guess) ###
import numpy as np
from quantum_engine.io import load_structure
from quantum_engine.io.atom_descriptor import resolve_atom, AtomTable
ts_pdb = f"{SCAN_DIR}cliff/cliff_ts.pdb"           # or any TS guess, e.g. neb_n25/ts.pdb
out = f"{REFINE_TS_DIR}freq_validate/"; Path(out).mkdir(parents=True, exist_ok=True)
# (1) auto-select the reacting region: substrate + nucleophile + metals + first shell of P/metals
_a, _bt, _ = load_structure(ts_pdb); _tb = AtomTable.from_biotite(_bt)
_resn = np.asarray(_bt.res_name); _pos = _a.get_positions()
_sel = set(np.where(np.isin(_resn, ['SUB','OHX','ZN']))[0].tolist())
for _c in [resolve_atom('SUB-P1', _tb, bare_int='serial')] + np.where(_resn=='ZN')[0].tolist():
    _sel |= set(np.where(np.linalg.norm(_pos-_pos[_c], axis=1) < 2.7)[0].tolist())
region = [f"0:{i}" for i in sorted(_sel)]
cmd = qcb_cmd('mace-polar-m', "freq", ts_pdb, "--model", "mace-polar-m", "--charge", 0,
              "--multiplicity", 1, "--device", "cuda", "--indices", *region, "--delta", 0.01, "--outdir", out)
print(" ".join(str(x) for x in cmd))
print(f"\n# region = {len(_sel)} atoms. AFTER it runs, validate the imaginary mode IS the reaction coordinate:")
print("#   f=np.load(out+'frequencies_cm.npy'); m=np.asarray(np.load(out+'modes.npy')[int(np.argmin(f))]).reshape(-1,3)")
print("#   require min(f) < -50 cm-1 AND that m has real amplitude on OHX-O3/SUB-P1/SUB-O7 (project onto O3->P & P->O7).")
print("#   A peripheral wobble with ~0 reactive-atom amplitude is a FALSE positive (we hit one: -65 cm-1, 0 reactive).")


# **STEP 9: Analysis — refined-TS verdict (1 imaginary mode?)**

In [ ]:
##################################################################
###  ANALYSIS: REFINED-TS VERDICT  (run after refine-ts)        ###
##################################################################
# Reads refine-ts summary.json: did we get a genuine first-order saddle?
# (exactly one imaginary mode below the cutoff, with enough reactive-atom overlap).

### INPUTS ###
refine_summary = f"{REFINE_TS_DIR}refine/summary.json"

### ANALYSIS ###
import json
_p = Path(refine_summary)
if not _p.is_file():
    print(f"# no summary.json yet at {refine_summary} — run refine-ts first.")
else:
    r = json.loads(_p.read_text())
    verdict = "PASS ✓" if r.get("overall_pass") else "FAIL ✗"
    print(f"refine-ts verdict        : {verdict}")
    print(f"n_imag (significant)     : {r.get('n_imag')}   (want exactly 1)")
    print(f"imaginary frequency      : {r.get('imag_freq_cm')} cm^-1   (want < -50)")
    print(f"imag-mode reactive overlap: {r.get('imag_mode_overlap')}   (want >= 0.5; 0.7-0.8 for SN2)")
    print(f"E(TS)                    : {r.get('energy_eV')} eV ({r.get('energy_kcal_mol'):.2f} kcal/mol)")
    print(f"saddle backend used      : {r.get('backend_used')}")
    if not r.get("overall_pass"):
        print("\n# FAIL → inspect: wrong/insufficient imaginary mode, or a 2nd imag mode."
              " Try a better guess (CI-NEB), --backend auto, or check for a pentacoordinate intermediate.")


# **STEP 10: Validate the TS — tiered Hessian (`cowboy-qc validate-ts`)**

In [ ]:
##################################################################
###  VALIDATE-TS  (independent tiered Hessian validation)      ###
##################################################################
# Independent of refine-ts. Tier A = reactive-atom partial Hessian; Tier B = active-region
# Hessian (catches a hidden 2nd imag mode in the metal/water shell); Tier C = Lanczos full
# check. Confirms exactly one imaginary mode on the reaction coordinate.
# --reactive-atoms accept atom tokens: descriptor (OHX-O3), serial:N, or a 0-based index.

### INPUTS ###
ts_pdb = f"{REFINE_TS_DIR}refine/ts_refined.pdb"   # the refined TS

### OUTPUTS ###
out_dir = f"{TS_VALIDATION_DIR}validate/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### VALIDATE PARAMETERS ###
reactive_atoms   = ['OHX-O3', 'SUB-P1', 'SUB-O7']   # atom tokens (nucleophile, center P, leaving group)
tier             = "b"          # 'a' | 'b' | 'c' | 'all' | comma list
active_region    = None         # Tier B select spec, e.g. 'sphere 6.0 around resid 169' (None=auto by reactive atoms)
imag_cm_cutoff   = -50.0
imag_overlap     = 0.5
n_imag_expected  = 1

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_validate_ts"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(ts_pdb).is_file():
    raise FileNotFoundError(f"ts_pdb not found: {ts_pdb}  (run refine-ts first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "validate-ts", ts_pdb, "--outdir", out_dir, "--model", model,
              "--charge", charge, "--multiplicity", multiplicity, "--device", device,
              "--reactive-atoms", *map(str, reactive_atoms), "--tier", tier,
              "--imag-cm-cutoff", imag_cm_cutoff, "--imag-mode-min-overlap", imag_overlap,
              "--n-imag-expected", n_imag_expected)
if head:          cmd += ["--head", head]
if active_region: cmd += ["--active-region", active_region]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output dir: {out_dir}  (per-tier PASS/FAIL + frequencies)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '04:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 11: Verify IRC-like (`cowboy-qc verify-irc-like`)**

In [ ]:
##################################################################
###  VERIFY-IRC-LIKE  (TS connects reactant & product basins)  ###
##################################################################
# Displace +/- along the imaginary mode and relax both branches -> confirm the TS connects
# the intended reactant and product (two distinct lower basins). Needs the imag-mode vector
# from refine-ts/validate-ts (imag_mode.npy). For organophosphate hydrolysis, watch for a
# PENTACOORDINATE intermediate (s ~ 0, both bonds ~1.7 A) — that's a real stepwise mechanism,
# not a failure (then do two-step NEB: R->intermediate, intermediate->P).

### INPUTS ###
ts_pdb    = f"{REFINE_TS_DIR}refine/ts_refined.pdb"
imag_mode = f"{REFINE_TS_DIR}refine/imag_mode.npy"   # emitted by refine-ts / validate-ts

### OUTPUTS ###
out_dir = f"{TS_VALIDATION_DIR}irc_like/"

### ENERGY MODEL ###
model, head, device = 'mace-polar-m', None, "cuda"
charge, multiplicity = 0, 1

### IRC-LIKE PARAMETERS ###
displacement = 0.20             # A along the imag mode
fmax         = 0.05
max_steps    = 200
optimizer    = "lbfgs"          # lbfgs | bfgs | fire

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_verify_irc"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
for p in (ts_pdb, imag_mode):
    if not Path(p).is_file():
        raise FileNotFoundError(f"not found: {p}  (run refine-ts/validate-ts first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = qcb_cmd(model, "verify-irc-like", ts_pdb, "--imag-mode", imag_mode, "--outdir", out_dir,
              "--model", model, "--charge", charge, "--multiplicity", multiplicity, "--device", device,
              "--displacement", displacement, "--fmax", fmax, "--max-steps", max_steps,
              "--optimizer", optimizer)
if head: cmd += ["--head", head]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output dir: {out_dir}  (forward/back basins + delta-energies)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '04:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 12: Analysis — energy barrier + reaction energy + Eyring rate**

In [ ]:
##################################################################
###  ANALYSIS: ENERGY BARRIER + REACTION ENERGY + RATE          ###
##################################################################
# The headline number. Barrier height = E(TS) - E(reactant_min); reaction energy =
# E(product_min) - E(reactant_min). Reads the three opt/refine summary.json energies.
# Then an Eyring estimate of the rate constant k(T) from the barrier.

### INPUTS ###
ts_summary       = f"{REFINE_TS_DIR}refine/summary.json"
reactant_summary = f"{RELAX_MINIMIZE_DIR}endpoints/reactant/opt-summary.json"
product_summary  = f"{RELAX_MINIMIZE_DIR}endpoints/product/opt-summary.json"

### PARAMETERS ###
T = 298.15                       # K, for the Eyring rate

### ANALYSIS ###
import json, math
def _E(path):   # eV from a summary.json
    p = Path(path)
    return json.loads(p.read_text()).get("energy_eV") if p.is_file() else None
E_ts, E_r, E_p = _E(ts_summary), _E(reactant_summary), _E(product_summary)
EV2KCAL = 23.0605
if None in (E_ts, E_r):
    print("# need E(TS) and E(reactant_min). Missing:",
          [n for n, v in [("E_TS", E_ts), ("E_reactant", E_r)] if v is None],
          "— run refine-ts + min-endpoints first.")
else:
    dE_fwd = (E_ts - E_r) * EV2KCAL   # ELECTRONIC barrier ΔE‡ (no ZPE/thermal/entropy)
    print(f"forward barrier  ΔE‡(fwd) = {dE_fwd:8.2f} kcal/mol")
    if E_p is not None:
        dE_rxn = (E_p - E_r) * EV2KCAL
        dE_rev = (E_ts - E_p) * EV2KCAL
        print(f"reverse barrier  ΔE‡(rev) = {dE_rev:8.2f} kcal/mol")
        print(f"reaction energy  ΔE(rxn)  = {dE_rxn:8.2f} kcal/mol")
    # Eyring: k = (kB T / h) exp(-ΔG‡ / RT); here we use the ELECTRONIC ΔE‡ as a ΔG‡ proxy.
    kB, h, R = 1.380649e-23, 6.62607015e-34, 1.987204e-3   # J/K, J·s, kcal/mol/K
    k = (kB * T / h) * math.exp(-dE_fwd / (R * T))
    print(f"\nEyring k({T:.0f} K)         = {k:.3e} s^-1   (ΔE‡ used as ΔG‡ proxy; add ZPE/entropy for rigor)")
    print("# NOTE: an MLFF electronic barrier; for a publication number re-evaluate at DFT (Step: ORCA).")


# **STEP 13: *(optional, EXPERIMENTAL)* AEFM refine the TS guess (out-of-domain on metals)**

In [ ]:
##################################################################
###  OPTIONAL/EXPERIMENTAL: AEFM refine on the di-Zn guess       ###
##################################################################
# ⚠ EXPERIMENTAL + OUT OF DOMAIN. AEFM is trained on CHNO gas-phase organics; a di-Zn/P
# active site is OUTSIDE its training, so its weights for Zn/P are UNTRAINED and the
# refinement is UNVALIDATED. AEFM does NOT crash on metals (LEFTNet embeds Z<100) — unlike
# React-OT — so you CAN try it with --allow-out-of-domain to see if it helps the guess.
# The QM saddle+Hessian+IRC gate (refine-ts/validate-ts) remains the sole authority; treat
# any AEFM output as a guess to be re-refined, never as a result. Runs in the AEFM sidecar.

### INPUTS ###
guess_pdb = f"{SCAN_DIR}scan/ts_guess.pdb"     # the scan/refine TS guess (pdb)

### OUTPUTS ###
out_dir     = f"{GENERATIVE_DIR}aefm_opaa/"
guess_xyz   = f"{out_dir}ts_guess.xyz"          # AEFM reads xyz — converted below
refined_out = f"{out_dir}aefm_opaa_refined.xyz"

### MODEL ###
charge, multiplicity = 0, 1

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_aefm_experimental"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(guess_pdb).is_file():
    print(f"# TS-guess pdb not found yet: {guess_pdb}  (run scan/refine-ts first)")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
# (1) convert the pdb guess -> xyz (AEFM reads xyz); (2) ts-refine with --allow-out-of-domain
# (REQUIRED — Zn/P are out of AEFM's CHNO training). Two commands, run in order.
commands = []
conv_py = f"{out_dir}pdb2xyz.py"
Path(conv_py).write_text(
    f"from ase.io import read, write; write(r'{guess_xyz}', read(r'{guess_pdb}'))\n")
cmd_conv = [*APPTAINER(AEFM_SIF), "python", conv_py]
cmd_ref = sidecar_cmd(AEFM_SIF, "ts-refine", "--method", "aefm", "--ts-guess", guess_xyz,
                      "--charge", charge, "--multiplicity", multiplicity, "--allow-out-of-domain",
                      "--out", refined_out, "--outdir", out_dir)
commands.append(" ".join(str(x) for x in cmd_conv))
commands.append(" ".join(str(x) for x in cmd_ref))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Output: {refined_out}  (EXPERIMENTAL — re-refine + validate with the QM gate!)")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '02:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 2
cpus_per_task = '1'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '6g'         # g = Gigabytes
queue         = 'gpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = 'small'        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


# **STEP 14: (optional) DFT Reference — ORCA native NEB-TS**

In [ ]:
##################################################################
###  DFT REFERENCE  (cowboy-qc ts-entry --engine orca; native NEB-TS) ###
##################################################################
# For a publication-grade barrier, route the whole TS step to ORCA's native NEB-TS / OptTS
# via the QM-engine gateway. --no-execute writes the ORCA input + an sbatch wrapper WITHOUT
# running, so you can inspect/queue it. Set the functional/basis + resources in the engine
# config. CPU partition (ORCA is CPU/MPI).

### INPUTS ###
entry        = "reactant-product"
spec_path    = f"{REACTION_SPEC_DIR}reaction_spec.yaml"
reactant_pdb = f"{RELAX_MINIMIZE_DIR}relax/reactant.pdb"   # EDIT
product_pdb  = f"{RELAX_MINIMIZE_DIR}relax/product.pdb"    # EDIT

### OUTPUTS ###
out_dir = f"{DFT_DIR}orca_nebts/"

### DFT PARAMETERS ###
charge, multiplicity  = 0, 1
engine_method = "wB97X-D3/def2-TZVP"   # set in the ORCA engine config; shown here for reference
EXECUTE       = False           # False -> write ORCA input + wrapper, don't run

### COMMAND / SUBMIT FILE NAMES ###
commands_name      = f"{PROJECT_NAME}_dft_orca"
commands_file_path = os.path.join(CMDS_DIR, commands_name)

### SANITY CHECKS ###
if not Path(spec_path).is_file():
    raise FileNotFoundError(f"reaction spec not found: {spec_path}")
Path(out_dir).mkdir(parents=True, exist_ok=True)

### GENERATE COMMANDS ###
commands = []
cmd = [*APPTAINER(MAIN_SIF, gpu=False), *CLI, "ts-entry", "--entry", entry, "--reaction-spec", spec_path,
       "--engine", "orca", "--reactant", reactant_pdb, "--product", product_pdb,
       "--charge", str(charge), "--multiplicity", str(multiplicity), "--outdir", out_dir,
       ("--execute" if EXECUTE else "--no-execute")]
commands.append(" ".join(str(x) for x in cmd))
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print(f"# {len(commands)} command(s) → {commands_file_path}")
for c in commands: print("\n" + c)
print(f"\n# Outputs: {out_dir}  (ORCA NEB-TS job; method = {engine_method})")

##################################### ---- [SETUP BATCH JOBS] ---- #####################################
# ── core knobs (always set these) ────────────────────────
qtime         = '24:00:00'   # NOTE: cluster MinTime is 15min — anything shorter gets bumped
cmds_per_job  = 1
cpus_per_task = '4'          # bump for dataloaders / parallel images, etc. | DEFAULT SHOULD ALWAYS BE 1
memory        = '32g'         # g = Gigabytes
queue         = 'cpu'        # 'cpu' | 'gpu' | 'gpu-bf' | 'gpu-train'
job_name      = os.path.basename(commands_file_path)
submit_file   = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs      = math.ceil(sum(1 for _ in open(commands_file_path, "r")) / cmds_per_job)
# ── GPU targeting (optional — only relevant if queue is a GPU partition) ──
gpu_class     = None        # 'small' (A4000/A5000/B4000/4000Ada) | 'large' (A6000/L40S/A100) | 'h200'
constraint    = None        # e.g. 'A4000', 'B4000|A5000', 'Blackwell', 'UW'  (None = any in class) | gpu model/gen, cpu model, location; '&'/'|' ok
exclude_nodes = None        # e.g. ['g2702']  if a node is misbehaving
# ── requeue + resilience (defaults are sensible; leave them) ───────
requeue              = True
max_restarts         = 2    # up to 3 attempts total per array task
pre_timeout_seconds  = 45   # USR1 fires 45s before walltime → graceful kill + requeue
# ── multi-GPU / MPI (leave defaults for typical single-GPU work) ──
ntasks               = 1    # MPI ranks per array task; >1 auto-adds --nodes=1
gpus_per_task        = None # None = 1 on GPU partitions, 0 on cpu | only specify to get more gpus if needed
# ── escape hatch + force-redo ─────────────────────────
extra_sbatch = None         # list[str] of raw '#SBATCH ...' lines for things this API misses # e.g. ['#SBATCH --mail-type=FAIL']
force_redo   = False        # True = wipe {logs_dir}/progress/{job_name}_* first (markers only, NOT cmd outputs)
# ── submit ──────────────────────────────────
nb.submit_array_job(commands_file_path, qtime, cpus_per_task, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue,
    gpu_class=gpu_class, constraint=constraint, exclude_nodes=exclude_nodes, ntasks=ntasks, gpus_per_task=gpus_per_task,                   # GPU targeting & multi-GPU / MPI
    requeue=requeue, max_restarts=max_restarts, pre_timeout_seconds=pre_timeout_seconds, extra_sbatch=extra_sbatch, force_redo=force_redo,) # requeue / resilience & escape hatch


---
# 🤖 Autonomous run log (Claude — 2026-06-09)

Steps run while you slept. Each step's **command** is in its STEP cell above (click-run to reproduce).
Notebook cells now request **6g / 1 CPU / `gpu` queue** (2–4h walltimes) — lowered from the original
16g/2CPU/6h based on observed usage (relax/scan/refine-ts never approached even 8g). DFT cell kept at
32g/CPU. (The overnight autonomous runs themselves used `gpu-bf` to conserve priority; the cells stay on `gpu`.)

| step | what | job | result |
|---|---|---|---|
| 3 | relax (CA-frozen, `is_ts_guess` auto-pinned both reactive bonds) | `3729427` | ✅ \|F\|max 0.13 (constrained-min); geometry sane |
| 4 | bond-difference scan, production (n=16, fmax 0.05) → `scan/` | `3738249` | ✅ **fwd barrier 3.93 kcal/mol**, TS at s=−0.613 (frame 6) |
| 6 | minimize reactant + product endpoints (ca-only, fmax 0.03) | `3743249` | ✅ both converged — ΔE_rxn = −14.5 kcal/mol (exothermic) |
| 8 | refine-ts on the **scan** TS guess (saddle + Hessian gate) | `3865184` | Sella fix ✅ (converged fmax 0.012) but verdict **FAIL: n_imag=0** — slid to a near-reactant minimum |
| 7 | **frozen-GSM** path search (minimized endpoints, ca-only) → real MEP | `3869201` | ✅ 13 images, **barrier 17.84 kcal/mol** in **3.5 min** on a gpu-train large GPU |
| 8b | refine-ts on the **GSM** TS guess (image 6, late TS) | `3869750` | RUNNING (gpu-train large GPU) |

**STEP 4 — scan result.** Clean 16-frame run; the engine's new interior-max picker put the **TS guess at
s=−0.613 (frame 6)** — an *early, reactant-like* TS (P–O7 breaking bond still shorter than the P–O3 forming
bond), which is exactly what Hammond's postulate predicts for an exothermic step. **Forward barrier ≈ 3.9
kcal/mol** (scan estimate; refine-ts gives the true saddle), reaction **exothermic by ~15 kcal/mol** —
consistent with efficient di-Zn phosphotriesterase catalysis. Wrote `ts_guess.pdb` + `reactant_scan.pdb` +
`product_scan.pdb`. (Two scan bugs were found+fixed during the coarse first pass — see below.)

**STEP 6 — endpoint minimization — DONE.** reactant_min (E=−303003.0355 eV, |F|max 0.030, 82 steps) +
product_min (E=−303003.6632 eV, 3 steps). ΔE_rxn = **−14.5 kcal/mol** (exothermic); preliminary barrier
from the true reactant min to the scan TS guess ≈ **4.4 kcal/mol** (refine-ts gives the rigorous saddle).
(reactive bonds free) so the barrier compares two real stationary points. Run as 2 parallel array tasks.

**STEP 7 — path search (the real barrier). KEY FINDING:** the frozen-GSM double-ended search (now
constraint-aware — your feature) gave a clean MEP: barrier **17.84 kcal/mol**, product **−14.47 kcal/mol**
(matching the independent endpoint ΔE of −14.5 — internal consistency ✓). The 1-D scan's **3.93 kcal/mol
was a severe underestimate** — relaxing every orthogonal DOF at each CV value let the system slip *around*
the barrier; the double-ended MEP climbs it. The GSM TS sits on the product side (O3–P made 1.73 Å, P–O7
departing 2.81 Å) — the rate-limiting leaving-group step. Fed to refine-ts (`3869750`). GSM ran in 3.5 min
on a gpu-train large GPU (vs ~min on small). CI-NEB (`3865666`) running as a second opinion.

**STEP 8 — refine-ts on the scan guess: the Sella fix works, but the 1-D scan guess isn't good enough.**
After the eigh fix, **Sella converged cleanly** (fmax 0.0122, backend=sella, ~35 min). But the partial
Hessian found **0 imaginary modes** (lowest +125.6 cm⁻¹, 56%% reactive overlap) and E=−303003.025 eV —
i.e. Sella relaxed *downhill* from the scan guess (E=−303002.843) to a minimum near the reactant
(E=−303003.0355), not the saddle. This is exactly the failure mode you flagged: a 1-D relaxed scan can
sit *beside* the true saddle, so the saddle search falls off it. The acceptance gate correctly returned
FAIL. → The **double-ended routes** (CI-NEB + frozen-GSM, both in flight) relax all orthogonal DOFs and
should give a TS guess that actually sits on the saddle; feed that to refine-ts.
Hessian on `OHX-O3/SUB-P1/SUB-O7` → require 1 imaginary mode < −50 cm⁻¹ with ≥50% reactive overlap. A
subagent is reviewing the acceptance logic before I launch the (expensive) saddle run.

**Engine fixes this session (all tested; suite 322 green, 0 regressions):**
- `scan_modes`: TS guess = highest **interior** maximum + forward barrier `E(TS)−E(reactant basin)` +
  `barrierless` flag (global-max picked strained endpoints for downhill reactions). +3 unit tests.
- scan **extract** crash fixed (eager-eval of `get_potential_energy()` default); template de-fragilized.
- `opt --optimizer` registry-validated (precon-lbfgs/fire2/torch-sim selectable; precon unsuitable for
  isolated clusters → lbfgs/fire here).
- STEP 5 analysis cell rewritten (glob outputs, read engine TS index/barrier). Notebook⇄generator byte-identical.
- **refine-ts** (subagent-reviewed CORRECT): made it stamp `info['spin']`=M (open-shell consistency with
  `opt`; saddle cascade now carries it) + honor the fix-preset's excluded residues (matters for `backbone*`);
  fixed the `spin` docstring. Acceptance gate (eigh order, cm⁻¹, n_imag<−50, normalized reactive overlap) verified.
- **Sella saddle bug FIXED** (blocked STEP 8): `_patched_eigh` forced a standard-only LAPACK driver
  (`evd`) onto *every* `scipy.linalg.eigh` call, but Sella also makes generalized `eigh(A,B)` calls →
  `ValueError: evd does not accept input b array`, which crashed both Sella backends so the search fell
  to an unseeded dimer that didn't converge. Now the driver is forced only for standard calls; +3 tests.
- **Frozen-atom GSM/FSM IMPLEMENTED** (your request): `cowboy-qc gsm` now takes `--fix-preset`/`--fix`/`--free`
  (+`--multiplicity`). Threaded a `freeze_atoms` index list through `_atoms_to_geom` → the endpoint pysisyphus
  geometries; `GrowingString` grows nodes via `Geometry.copy()` which preserves `freeze_atoms`, so every string
  node inherits the CA-freeze. Validated end-to-end (EMT GSM: frozen atom moved 0.000000 Å across all images,
  free atom moved 1.12 Å). +4 tests (`test_gsm_freeze.py`). Path-search cell now passes `--fix-preset` for GSM/FSM.

_(further entries appended as steps complete)_

## 🧭 MECHANISM MAP (mace-polar-m, di-Zn OPAA)
Cross-validating CI-NEB + frozen-GSM revealed a **stepwise associative** mechanism. Refining each path
method's peak relaxed it into a basin (Sella-Cartesian minimizes from an in-basin guess), which *mapped the
stationary minima* even though it didn't pin saddles:

| stationary point | E (kcal/mol vs reactant) | geometry (O3–P / P–O7) |
|---|---|---|
| reactant | 0.00 | 3.x / 1.6 (substrate intact) |
| **pentacoordinate intermediate** | **−5.7** | 1.66 / 2.01 (phosphorane, di-Zn-stabilized) |
| post-cleavage complex | −6.2 | 1.57 / 4.97 (O7 expelled) |
| product | −14.5 | 1.5 / >5 (relaxed product) |

The di-Zn site **stabilizes a pentacoordinate phosphorane below the reactant** — the textbook PTE mechanism.
**TS-search lesson:** path-method peaks sit in basins here → the Sella-first cascade minimizes and 'succeeds'
before the mode-seeded dimer runs. **Fix in flight:** segment NEBs between *adjacent* minima (reactant→inter =
`3876230`, inter→product = `3876231`, gpu-train large GPUs) give on-barrier climbing images → refine those
(--from-neb, dimer if needed) to pin the **addition** + **elimination** TSs. Rate-limiting = the higher of the two.
View any structure: `output/ts_views/` (active-site PDBs+PNGs) or the py3Dmol snippet.


---
## 📦 Deliverables — clean, stem-named copies of the key structures
Collects the pipeline's important outputs into `output/deliverables/` with a **`{PROJECT_NAME}_<role>.pdb`**
naming convention (stem derived from the initial theozyme input), so they're unambiguous and easy to hand off.


In [ ]:
### DELIVERABLES — copy key outputs to clean {PROJECT_NAME}_<role>.pdb names ###
import shutil
STEM  = PROJECT_NAME                                    # e.g. 'opaa_theozyme' (from the initial input)
deliv = Path(RELAX_MINIMIZE_DIR).parent / "deliverables"; deliv.mkdir(parents=True, exist_ok=True)
mapping = {
    f"{RELAX_MINIMIZE_DIR}relax/relaxed.pdb":                   f"{STEM}_ca_frozen_relaxed.pdb",
    f"{RELAX_MINIMIZE_DIR}endpoints/reactant/reactant_min.pdb": f"{STEM}_minimized_reactant.pdb",
    f"{RELAX_MINIMIZE_DIR}endpoints/product/product_min.pdb":   f"{STEM}_minimized_product.pdb",
    f"{SCAN_DIR}scan/ts_guess.pdb":                             f"{STEM}_scan_ts_guess.pdb",
    f"{PATH_SEARCH_DIR}neb_n25/ts.pdb":                         f"{STEM}_neb_ts_guess_pentacoordinate.pdb",
}
for src, name in mapping.items():
    if Path(src).is_file():
        shutil.copy(src, deliv / name); print(f"  {name:48s} <-  {src}")
print(f"\n# clean deliverables in {deliv}/  (the pentacoordinate NEB guess is the best TS structure -> DFT/QM-MM)")


---
## 📊 Pipeline schematic (auto-generated)
Parses **this notebook's** STEP cells and draws the flow (core = solid blue, optional = dashed grey).
Edit/add steps and re-run — it regenerates from whatever it detects.


In [ ]:
import sys; sys.path.insert(0, "../lib")          # notebooks/lib (relative to opaa_theozyme/)
from pipeline_schematic import render_pipeline
render_pipeline("opaa_ts_pipeline.ipynb", out_png="opaa_pipeline_schematic.png",
                show=True, title="OPAA di-Zn — TS pipeline")
